In [ ]:
# =========================================
# INSTALL
# =========================================
!pip install networkx scikit-learn xgboost


# =========================================
# IMPORTS
# =========================================
import pandas as pd
import numpy as np
import networkx as nx
from collections import Counter
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import KFold


# =========================================
# LOAD DATA
# =========================================
print("\n[INFO] Loading data...")
df = pd.read_csv("processed_can_data.csv")
ref_df = pd.read_csv("normal_reference_values.csv")

df["CAN_ID"] = df["CAN_ID"].astype(str).str.strip()
df["Flag"] = df["Flag"].astype(str).str.strip()

print("[INFO] Data Loaded:", df.shape)


# =========================================
# PARAMETERS
# =========================================
window_sizes = [50, 100, 500, 1000]
alpha = 0.02

paper_thresholds = {
    50: {"gamma1": 5.0, "gamma2": 5.0},
    100: {"gamma1": 3.0, "gamma2": 4.3},
    500: {"gamma1": 1.8, "gamma2": 2.5},
    1000: {"gamma1": 1.1, "gamma2": 1.6}
}

gamma3_values = {
    50: 4.4,
    100: 2.8,
    500: 3.2,
    1000: 3.7
}


# =========================================
# SPLIT FIRST (NO LEAKAGE)
# =========================================
print("\n[INFO] Splitting dataset...")
split_idx = int(0.7 * len(df))

train_df = df.iloc[:split_idx]
test_df  = df.iloc[split_idx:]

print("[INFO] Train:", train_df.shape)
print("[INFO] Test :", test_df.shape)


# =========================================
# ENTROPY
# =========================================
def calculate_entropy(values):
    counts = Counter(values)
    total = len(values)
    probs = [c/total for c in counts.values()]
    return -sum(p*np.log2(p) for p in probs if p > 0)


# =========================================
# GRAPH FEATURES
# =========================================
def build_graph(can_ids):
    G = nx.DiGraph()
    for i in range(len(can_ids)-1):
        G.add_edge(can_ids[i], can_ids[i+1])
    return G


def graph_features(can_ids):
    G = build_graph(can_ids)

    nodes = G.number_of_nodes()
    edges = G.number_of_edges()

    in_deg = dict(G.in_degree())
    out_deg = dict(G.out_degree())

    max_in = max(in_deg.values()) if in_deg else 0
    min_in = min(in_deg.values()) if in_deg else 0
    max_out = max(out_deg.values()) if out_deg else 0
    min_out = min(out_deg.values()) if out_deg else 0

    pr = nx.pagerank(G) if nodes > 0 else {}
    pr_vals = list(pr.values()) if pr else [0]

    pr_max = max(pr_vals)
    pr_min = min(pr_vals)
    pr_median = np.median(pr_vals)

    return [
        nodes, edges,
        max_in, max_out,
        min_in, min_out,
        pr_max, pr_min, pr_median
    ]


# =========================================
# STATISTICAL FEATURES
# =========================================
def stat_features(window, ref, gamma1, gamma2, gamma3):
    can_ids = window["CAN_ID"].tolist()

    freq = Counter(can_ids).values()

    mean = np.mean(list(freq))
    std = np.std(list(freq))
    entropy = calculate_entropy(can_ids)

    mean_score = abs(mean - ref["mu_a"]) / (ref["sigma_a"] + 1e-9)
    std_score  = abs(std  - ref["mu_s"]) / (ref["sigma_s"] + 1e-9)
    ent_score  = abs(entropy - ref["mu_e"]) / (ref["sigma_e"] + 1e-9)

    v1 = int(mean_score > gamma1)
    v2 = int(std_score > gamma2)
    v3 = int(ent_score > gamma3)

    score = v1 + v2 + v3

    return mean, std, entropy, score


# =========================================
# MAIN LOOP PER WINDOW SIZE
# =========================================
for w in window_sizes:

    print("\n" + "="*60)
    print(f"[INFO] PROCESSING WINDOW SIZE = {w}")
    print("="*60)

    ref = ref_df[ref_df["Window_Size"] == w].iloc[0]

    gamma1 = paper_thresholds[w]["gamma1"]
    gamma2 = paper_thresholds[w]["gamma2"]
    gamma3 = gamma3_values[w]

    print(f"[INFO] gamma1={gamma1}, gamma2={gamma2}, gamma3={gamma3}")

    def create_dataset(data, name):

        X = []
        y = []

        for start in range(0, len(data), w):
            end = start + w
            if end > len(data):
                break

            window = data.iloc[start:end]

            attack_ratio = (window["Flag"] == "T").sum() / w
            label = 1 if attack_ratio >= alpha else 0

            mean, std, entropy, score = stat_features(
                window, ref, gamma1, gamma2, gamma3
            )

            g_feats = graph_features(window["CAN_ID"].tolist())

            features = [mean, std, entropy, score] + g_feats

            X.append(features)
            y.append(label)

        return np.array(X), np.array(y)

    X_train_full, y_train_full = create_dataset(train_df, "TRAIN")
    X_test, y_test   = create_dataset(test_df, "TEST")

    # -------------------------------
    # CROSS VALIDATION
    # -------------------------------
    kf = KFold(n_splits=5, shuffle=True, random_state=42)

    for train_index, val_index in kf.split(X_train_full):

        X_train_fold, X_val_fold = X_train_full[train_index], X_train_full[val_index]
        y_train_fold, y_val_fold = y_train_full[train_index], y_train_full[val_index]

        model_cv = XGBClassifier(use_label_encoder=False, eval_metric='logloss')
        model_cv.fit(X_train_fold, y_train_fold)

        y_pred_cv = model_cv.predict(X_val_fold)

    # -------------------------------
    # FINAL MODEL
    # -------------------------------
    model = XGBClassifier(use_label_encoder=False, eval_metric='logloss')
    model.fit(X_train_full, y_train_full)

    y_pred = model.predict(X_test)

    print(confusion_matrix(y_test, y_pred))
    print(classification_report(y_test, y_pred))


[INFO] Loading data...
[INFO] Data Loaded: (3500000, 2)

[INFO] Splitting dataset...
[INFO] Train: (2450000, 2)
[INFO] Test : (1050000, 2)

[INFO] PROCESSING WINDOW SIZE = 50
[INFO] gamma1=5.0, gamma2=5.0, gamma3=4.4


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [08:11:43] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [08:11:44] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [08:11:45] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


[[19214     3]
 [   16  1767]]
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     19217
           1       1.00      0.99      0.99      1783

    accuracy                           1.00     21000
   macro avg       1.00      1.00      1.00     21000
weighted avg       1.00      1.00      1.00     21000


[INFO] PROCESSING WINDOW SIZE = 100
[INFO] gamma1=3.0, gamma2=4.3, gamma3=2.8


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [08:14:02] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [08:14:03] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


[[9597    2]
 [   7  894]]
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      9599
           1       1.00      0.99      0.99       901

    accuracy                           1.00     10500
   macro avg       1.00      1.00      1.00     10500
weighted avg       1.00      1.00      1.00     10500


[INFO] PROCESSING WINDOW SIZE = 500
[INFO] gamma1=1.8, gamma2=2.5, gamma3=3.2


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [08:14:32] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


[[1902    0]
 [   0  198]]
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      1902
           1       1.00      1.00      1.00       198

    accuracy                           1.00      2100
   macro avg       1.00      1.00      1.00      2100
weighted avg       1.00      1.00      1.00      2100


[INFO] PROCESSING WINDOW SIZE = 1000
[INFO] gamma1=1.1, gamma2=1.6, gamma3=3.7


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [08:14:47] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


[[940   0]
 [  0 110]]
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       940
           1       1.00      1.00      1.00       110

    accuracy                           1.00      1050
   macro avg       1.00      1.00      1.00      1050
weighted avg       1.00      1.00      1.00      1050



In [ ]:
import joblib
import numpy as np
from collections import Counter
import networkx as nx
from xgboost import XGBClassifier
from sklearn.model_selection import KFold

# Set the window size for the model to be downloaded
w = 100

print(f"\n[INFO] Retraining model for window size = {w} for download...")

# Re-fetch parameters for this specific window size
ref = ref_df[ref_df["Window_Size"] == w].iloc[0]

gamma1 = paper_thresholds[w]["gamma1"]
gamma2 = paper_thresholds[w]["gamma2"]
gamma3 = gamma3_values[w]

# Re-create the dataset for this window size using functions defined in HXo0TPFQKJLC
X_train_full, y_train_full = create_dataset(train_df, "TRAIN")
X_test, y_test   = create_dataset(test_df, "TEST")

# Train the final model for this specific window size
# (Cross-validation is skipped here for brevity, as the user only wants the final model instance for download)
model_to_save = XGBClassifier(use_label_encoder=False, eval_metric='logloss')
model_to_save.fit(X_train_full, y_train_full)

# Save the trained model to a file
model_filename = f'xgboost_model_window_{w}.joblib'
joblib.dump(model_to_save, model_filename)
print(f"Model for window size {w} saved as {model_filename}")

# Provide code to download the file
from google.colab import files
files.download(model_filename)


[INFO] Retraining model for window size = 100 for download...
Model for window size 100 saved as xgboost_model_window_100.joblib


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [09:10:01] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
print("Downloading model...")
# !wget -O your_model.h5 "https://example.com/path/to/your_model.h5" # Uncomment and replace with your model URL
print("Model download complete.")

Model download complete.
